In [ ]:
import numpy as np
import pandas as pd
import importlib
from pathlib import Path

import data_pipeline
importlib.reload(data_pipeline)
from data_pipeline import DataPipeline, load_data, train_test_split

## Task A1: EDA khảo sát và chẩn đoán

In [ ]:
data_path = Path("data/melb_data.csv")
df = load_data(data_path)
df.head()

In [ ]:
df.info()

ghi chú: Schema gồm numeric và categorical; `Price` là target, các bước sửa dữ liệu chưa được áp dụng ở block này.


In [ ]:
df.describe()

ghi chú: Các cột numeric có đơn vị và biên độ khác nhau; pipeline cần chuẩn hóa Z-score sau khi encode để đưa feature về cùng hệ đo.


In [ ]:
missing_values = pd.DataFrame(
    {
        "missing_count": df.isna().sum(),
        "missing_pct": df.isna().mean().mul(100).round(2),
    }
).query("missing_count > 0").sort_values("missing_pct", ascending=False)
missing_values

ghi chú: Missing tập trung ở `BuildingArea`, `YearBuilt`, `CouncilArea`, `Car`; numeric missing cần xử lý sau split bằng fitted imputer, `CouncilArea` không đưa vào model matrix theo contract hiện tại.


In [ ]:
outlier_columns = ["Price", "Landsize", "BuildingArea", "YearBuilt"]
outlier_detection = pd.DataFrame(
    [
        {"check": "Price > 99th percentile", "rows": int((df["Price"] > df["Price"].quantile(0.99)).sum())},
        {"check": "Landsize == 0", "rows": int((df["Landsize"] == 0).sum())},
        {"check": "Landsize > 10000", "rows": int((df["Landsize"] > 10000).sum())},
        {"check": "BuildingArea <= 0", "rows": int((df["BuildingArea"] <= 0).sum())},
        {"check": "BuildingArea > 1000", "rows": int((df["BuildingArea"] > 1000).sum())},
        {"check": "YearBuilt < 1800", "rows": int((df["YearBuilt"] < 1800).sum())},
    ]
)
outlier_detection

ghi chú: `BuildingArea <= 0`, `YearBuilt < 1800`, `Landsize <= 0` không phù hợp về mặt giá trị đo; pipeline chuyển các giá trị này thành missing trước khi impute hoặc tính ratio.


ghi chú: A1 chỉ khảo sát dữ liệu thô; chưa sửa missing value, invalid value, scale hoặc encode.


## Task A2: Chia tách dữ liệu thô

In [ ]:
X_train_raw, X_test_raw = train_test_split(df, test_size=0.3, random_state=42)

split_summary = pd.DataFrame(
    {
        "split": ["X_train_raw", "X_test_raw"],
        "rows": [len(X_train_raw), len(X_test_raw)],
        "pct": [round(len(X_train_raw) / len(df) * 100, 2), round(len(X_test_raw) / len(df) * 100, 2)],
    }
)
split_summary

ghi chú: Split được thực hiện trên DataFrame thô; mọi trạng thái fit của imputer, encoder và scaler chỉ học từ `X_train_raw`.


## EDA trên train split


In [ ]:
target_distribution_check = pd.DataFrame(
    {
        "X_train_raw": X_train_raw["Price"].describe(percentiles=[0.05, 0.5, 0.95]),
        "X_test_raw": X_test_raw["Price"].describe(percentiles=[0.05, 0.5, 0.95]),
    }
).round(2)
target_distribution_check


ghi chú: `Price` không thiếu giá trị; `y_train` và `y_test` được tách trực tiếp từ split thô, không biến đổi target trong pipeline.


In [ ]:
train_missing_summary = pd.DataFrame(
    {
        "missing_count": X_train_raw.isna().sum(),
        "missing_pct": X_train_raw.isna().mean().mul(100).round(2),
    }
).query("missing_count > 0").sort_values("missing_pct", ascending=False)
train_missing_summary


ghi chú: Missing numeric được xử lý bằng `KNNImputer` fit trên `X_train_raw`; các cột thiếu nhiều như `BuildingArea` và `YearBuilt` có thêm missing flag trước imputation.


In [ ]:
train_invalid_summary = pd.DataFrame(
    [
        {"check": "Price <= 0", "rows": int((X_train_raw["Price"] <= 0).sum())},
        {"check": "Landsize <= 0", "rows": int((X_train_raw["Landsize"] <= 0).sum())},
        {"check": "Landsize > 10000", "rows": int((X_train_raw["Landsize"] > 10000).sum())},
        {"check": "BuildingArea <= 0", "rows": int((X_train_raw["BuildingArea"] <= 0).sum())},
        {"check": "BuildingArea > 1000", "rows": int((X_train_raw["BuildingArea"] > 1000).sum())},
        {"check": "YearBuilt < 1800", "rows": int((X_train_raw["YearBuilt"] < 1800).sum())},
    ]
)
train_invalid_summary


ghi chú: Outlier giá trị lớn chỉ dùng để nhận diện phân phối lệch; pipeline không drop row outlier, chỉ repair các giá trị không hợp lệ đã xác định.


In [ ]:
numeric_columns = [
    "Price",
    "Rooms",
    "Bedroom2",
    "Bathroom",
    "Car",
    "Distance",
    "Landsize",
    "BuildingArea",
    "YearBuilt",
    "Propertycount",
]

numeric_profile = X_train_raw[numeric_columns].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T
numeric_profile["skew"] = X_train_raw[numeric_columns].skew(numeric_only=True)
numeric_profile.round(2)


ghi chú: `Landsize`, `BuildingArea`, `Distance` có scale lệch mạnh so với các biến đếm; bước scaling phải chạy sau imputation, feature engineering và one-hot để chuẩn hóa toàn bộ ma trận cuối.


In [ ]:
relationship_columns = ["Price", "Rooms", "Bedroom2", "Bathroom", "Car", "Distance", "BuildingArea", "YearBuilt"]
relationship_summary = X_train_raw[relationship_columns].corr(numeric_only=True)[["Price", "Rooms"]].round(3)
relationship_summary


ghi chú: `Bedroom2` chồng lấp với `Rooms`; cột này được đưa vào `drop_columns`, còn `Rooms`, `Bathroom` và `OtherRooms` giữ lại tín hiệu quy mô và bố cục.


In [ ]:
categorical_columns = ["Type", "Regionname", "Method", "CouncilArea"]
categorical_summary = pd.DataFrame(
    {
        "column": categorical_columns,
        "unique_values": [X_train_raw[column].nunique(dropna=True) for column in categorical_columns],
        "missing_pct": [round(X_train_raw[column].isna().mean() * 100, 2) for column in categorical_columns],
        "top_value": [X_train_raw[column].mode(dropna=True).iloc[0] for column in categorical_columns],
    }
)
categorical_summary


ghi chú: `Type` và `Regionname` có cardinality thấp, phù hợp one-hot; `Method` và `CouncilArea` bị drop để giữ đúng phạm vi encoding của task A4.


In [ ]:
feature_preview = X_train_raw[["Rooms", "Bathroom", "BuildingArea", "Landsize", "YearBuilt"]].copy()
feature_preview["Age"] = 2026 - feature_preview["YearBuilt"]
feature_preview["OtherRooms"] = (feature_preview["Rooms"] - feature_preview["Bathroom"]).clip(lower=0)
feature_preview["BuildingArea_per_Room"] = feature_preview["BuildingArea"] / feature_preview["Rooms"].replace(0, np.nan)
feature_preview["BuildingCoverage"] = feature_preview["BuildingArea"] / feature_preview["Landsize"].replace(0, np.nan)
feature_preview[["Age", "OtherRooms", "BuildingArea_per_Room", "BuildingCoverage"]].describe().T.round(2)


ghi chú: Feature tạo thêm chỉ dùng phép biến đổi đơn giản từ cột có sẵn; các ratio phát sinh NaN hoặc inf sẽ được xử lý bằng fallback numeric đã fit từ train.


## Tasks A3-A6: Thực thi pipeline

In [ ]:
pipeline = DataPipeline(drop_columns=["Bedroom2"])
X_train, y_train = pipeline.fit_transform(X_train_raw)
X_test, y_test = pipeline.transform(X_test_raw)

metadata = {
    "feature_names": pipeline.feature_names,
    "target_name": pipeline.target_name,
    "pipeline": pipeline,
    "imputation_strategy": "knn_imputer",
    "scaling_method": "standardize",
    "encoded_categorical_columns": pipeline.categorical_columns,
    "encoded_columns": pipeline.encoded_columns,
    "drop_columns": pipeline.drop_columns,
    "engineered_features": [
        "Age",
        "OtherRooms",
        "BuildingArea_per_Room",
        "BuildingCoverage",
        "BuildingArea_missing",
        "YearBuilt_missing",
        "Landsize_zero_or_missing",
    ],
}

pd.DataFrame(
    {
        "artifact": ["X_train", "X_test", "y_train", "y_test", "features"],
        "shape": [X_train.shape, X_test.shape, y_train.shape, y_test.shape, len(metadata["feature_names"])],
    }
)

## Feature contract validation

In [ ]:
expected_engineered_features = metadata["engineered_features"]
X_train_df = pd.DataFrame(X_train, columns=metadata["feature_names"])
X_test_df = pd.DataFrame(X_test, columns=metadata["feature_names"])

feature_contract_check = pd.DataFrame(
    {
        "feature": expected_engineered_features,
        "exists_in_output": [feature in metadata["feature_names"] for feature in expected_engineered_features],
        "train_has_no_nan": [
            bool(not X_train_df[feature].isna().any()) if feature in X_train_df.columns else False
            for feature in expected_engineered_features
        ],
        "test_has_no_nan": [
            bool(not X_test_df[feature].isna().any()) if feature in X_test_df.columns else False
            for feature in expected_engineered_features
        ],
        "train_is_finite": [
            bool(np.isfinite(X_train_df[feature]).all()) if feature in X_train_df.columns else False
            for feature in expected_engineered_features
        ],
        "test_is_finite": [
            bool(np.isfinite(X_test_df[feature]).all()) if feature in X_test_df.columns else False
            for feature in expected_engineered_features
        ],
    }
)

failed_feature_contract = feature_contract_check.loc[
    ~feature_contract_check.drop(columns=["feature"]).all(axis=1)
]
assert failed_feature_contract.empty, failed_feature_contract.to_string(index=False)
feature_contract_check

## Handoff validation

In [ ]:
handoff_checks = pd.DataFrame(
    [
        {"check": "X_train has no NaN", "passed": bool(not np.isnan(X_train).any())},
        {"check": "X_test has no NaN", "passed": bool(not np.isnan(X_test).any())},
        {"check": "X_train is finite", "passed": bool(np.isfinite(X_train).all())},
        {"check": "X_test is finite", "passed": bool(np.isfinite(X_test).all())},
        {"check": "same feature count", "passed": bool(X_train.shape[1] == X_test.shape[1])},
        {"check": "metadata feature count", "passed": bool(len(metadata["feature_names"]) == X_train.shape[1])},
        {"check": "y_train has no NaN", "passed": bool(not np.isnan(y_train).any())},
        {"check": "y_test has no NaN", "passed": bool(not np.isnan(y_test).any())},
        {"check": "y_train is finite", "passed": bool(np.isfinite(y_train).all())},
        {"check": "y_test is finite", "passed": bool(np.isfinite(y_test).all())},
        {"check": "X_train/y_train aligned", "passed": bool(X_train.shape[0] == len(y_train))},
        {"check": "X_test/y_test aligned", "passed": bool(X_test.shape[0] == len(y_test))},
    ]
)

assert handoff_checks["passed"].all()
handoff_checks

In [ ]:
pd.DataFrame({"feature_name": metadata["feature_names"]})